In [ ]:
# placeholder
# sources to pull 
https://www.weather.gov.sg/climate-historical-daily/
https://data.gov.sg/datasets?topics=environment&resultId=1459&page=1
https://www.met.gov.my/en/pencerapan/radar-malaysia/

https://www.weather.gov.sg/files/dailydata/DAILYDATA_S24_202508.csv

In [32]:
import requests
import pandas as pd
from io import StringIO
import numpy as np

In [28]:
url = 'https://www.weather.gov.sg/files/dailydata/DAILYDATA_S64_202508.csv'

header = { 
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.5 Safari/605.1.15",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
}

res = requests.get(url,headers = header)



print(res.status_code)
print(res.headers['Content-Type'])

df = pd.read_csv(StringIO(res.text))
df.head()



200
text/csv


,ï»¿Station,Year,Month,Day,Daily Rainfall Total (mm),Highest 30 min Rainfall (mm),Highest 60 min Rainfall (mm),Highest 120 min Rainfall (mm),Mean Temperature (Â°C),Maximum Temperature (Â°C),Minimum Temperature (Â°C),Mean Wind Speed (km/h),Max Wind Speed (km/h)
0,Bukit Panjang,2025,8,1,0.0,0.0,0.0,0.0,-,-,-,-,-
1,Bukit Panjang,2025,8,2,4.8,4.8,4.8,4.8,-,-,-,-,-
2,Bukit Panjang,2025,8,3,25.0,22.4,23.6,24.2,-,-,-,-,-
3,Bukit Panjang,2025,8,4,9.4,7.0,8.4,8.6,-,-,-,-,-
4,Bukit Panjang,2025,8,5,12.0,2.2,3.8,6.4,-,-,-,-,-


In [ ]:
# take windspeed , rainfall_mm, timestamp, station id 
df_filtered = df
df_filtered['timestamp'] = (df_filtered['Year'].astype(str))+'-'+df_filtered['Month'].astype(str).str.zfill(2) + '-'+df_filtered['Day'].astype(str).str.zfill(2)
df_filtered= df_filtered.drop(columns=['Year','Month','Day'])

# renaming all columns to be proper column names 
df_filtered = df_filtered.rename(columns={
    'ï»¿Station':'station_name'
    ,'Daily Rainfall Total (mm)' : 'daily_total_rainfall'
    ,'Highest 30 min Rainfall (mm)' : 'highest_30min_rainfall'
    ,'Highest 60 min Rainfall (mm)' : 'highest_60min_rainfall'
    ,'Highest 120 min Rainfall (mm)' : 'highest_120min_rainfall'
    ,'Mean Temperature (Â°C)' : 'mean_temperature'
    ,'Maximum Temperature (Â°C)': 'max_temperature'
    ,'Minimum Temperature (Â°C)' :'min_temperature'
    ,'Mean Wind Speed (km/h)' : 'mean_wind_speed'
    ,'Max Wind Speed (km/h)' : 'max_wind_speed'
    })

# adding station_id column for downstream table joining
df_filtered['station_name'] = df_filtered['station_name'].astype('string').str.strip().str.lower()
df_filtered['station_id'] = 'S64' # to replace dynamic 

# rearrrange columns 
df_filtered= df_filtered[['timestamp','station_name','station_id','daily_total_rainfall','highest_30min_rainfall','highest_60min_rainfall','highest_120min_rainfall','mean_temperature','max_temperature','min_temperature','mean_wind_speed','max_wind_speed']]

# replace - with nan to assert column type if not insert into database will throw error
df_filtered = df_filtered.replace('-',np.nan)

df_filtered.head()

/var/folders/c9/rl6pcdvd1_xb6ng3n8l9wr2w0000gn/T/ipykernel_38598/2272789227.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_filtered = df_filtered.replace('-',np.nan)


,timestamp,station_name,station_id,daily_total_rainfall,highest_30min_rainfall,highest_60min_rainfall,highest_120min_rainfall,mean_temperature,max_temperature,min_temperature,mean_wind_speed,max_wind_speed
0,2025-08-01,bukit panjang,S64,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,2025-08-02,bukit panjang,S64,4.8,4.8,4.8,4.8,NaN,NaN,NaN,NaN,NaN
2,2025-08-03,bukit panjang,S64,25.0,22.4,23.6,24.2,NaN,NaN,NaN,NaN,NaN
3,2025-08-04,bukit panjang,S64,9.4,7.0,8.4,8.6,NaN,NaN,NaN,NaN,NaN
4,2025-08-05,bukit panjang,S64,12.0,2.2,3.8,6.4,NaN,NaN,NaN,NaN,NaN


In [34]:
df_filtered.dtypes

timestamp                          object
station_name               string[python]
station_id                         object
daily_total_rainfall              float64
highest_30min_rainfall            float64
highest_60min_rainfall            float64
highest_120min_rainfall           float64
mean_temperature                  float64
max_temperature                   float64
min_temperature                   float64
mean_wind_speed                   float64
max_wind_speed                    float64
dtype: object